# Profile the Sun Model with SPECTRV

This notebook reuses the ATLAS solar atmosphere in `~/ATLAS_sun_model` and runs its SYNTHE step with `bin/profile/spectrv.profile.exe`. Other SYNTHE executables remain unchanged. The default wavelength range and resolution match `sun_model.ipynb`; the run can take a while.

Run the structure section of `sun_model.ipynb` first if the solar atmosphere files are not present. This notebook does not delete or recalculate that model.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

candidate_roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'BasicATLAS', Path.home() / 'BasicATLAS']
project_root = next((path.resolve() for path in candidate_roots if (path / 'atlas.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Could not locate BasicATLAS/atlas.py from the notebook working directory')
sys.path.insert(0, str(project_root))

import atlas

run_dir = Path.home() / 'ATLAS_sun_model'
profile_exe = project_root / 'bin' / 'profile' / 'spectrv.profile.exe'
required_model_files = [run_dir / 'output_last_iteration.out', run_dir / 'output_summary.out']
missing_model_files = [str(path) for path in required_model_files if not path.is_file()]
if missing_model_files:
    raise FileNotFoundError('Solar ATLAS output is missing: ' + ', '.join(missing_model_files) + '. Run the structure section of sun_model.ipynb first.')
if not profile_exe.is_file() or not os.access(profile_exe, os.X_OK):
    raise FileNotFoundError(f'SPECTRV profiling executable is missing or not executable: {profile_exe}')
if shutil.which('gprof') is None:
    raise FileNotFoundError('gprof is required; install the GNU binutils package and retry')

print(f'ATLAS model: {run_dir}')
print(f'Profiled SPECTRV: {profile_exe}')

ATLAS model: /home/bbzze2/ATLAS_sun_model
Profiled SPECTRV: /home/bbzze2/BasicATLAS/bin/profile/spectrv.profile.exe


## Run the profiled synthesis

The `atlas.synthe` function writes and launches its normal batch script. This cell temporarily redirects only that script's SPECTRV command to the instrumented executable, then restores the template. It leaves `gmon.out` in each `synthe_N` batch directory.

In [3]:
min_wl = 100
max_wl = 3000
resolution = 600000.0
normal_spectrv_command = '{synthe_suite}/spectrv.exe'
original_template = atlas.templates.synthe_control

if original_template.count(normal_spectrv_command) != 1:
    raise RuntimeError('Expected exactly one SPECTRV command in the SYNTHE launch template; no run was started')

atlas.templates.synthe_control = original_template.replace(normal_spectrv_command, str(profile_exe))
try:
    atlas.synthe(run_dir, min_wl, max_wl, res=resolution)
finally:
    atlas.templates.synthe_control = original_template

Adapted the ATLAS-9 model to SYNTHE in output_synthe.out
Launcher created for wavelength range (100, 2850), batch 1. Expected number of points: 2009944 (buffer 2010001)


199.0it [08:12,  2.47s/it]                        


SYNTHE halted
ATLAS requested/received validation for /home/bbzze2/ATLAS_sun_model successful
SYNTHE requested/received validation for /home/bbzze2/ATLAS_sun_model successful
Launcher created for wavelength range (2850, 3000), batch 2. Expected number of points: 30776 (buffer 2010001)


199.0it [00:38,  5.23it/s]                        

SYNTHE halted
ATLAS requested/received validation for /home/bbzze2/ATLAS_sun_model successful
SYNTHE requested/received validation for /home/bbzze2/ATLAS_sun_model successful
Finished running SYNTHE in 0:08:50.262201 s


## Generate gprof reports

Each wavelength batch has a separate `gmon.out`, so this cell writes one report per batch next to its profile data.

In [4]:
gmon_files = sorted(run_dir.glob('synthe_*/gmon.out'))
if not gmon_files:
    raise FileNotFoundError(f'No gmon.out files found under {run_dir}. Confirm the profiled synthesis completed successfully.')

report_paths = []
for gmon_file in gmon_files:
    result = subprocess.run(
        ['gprof', str(profile_exe), str(gmon_file)],
        capture_output=True,
        text=True,
        check=True,
    )
    report_file = gmon_file.with_name('gprof_spectrv.txt')
    report_file.write_text(result.stdout)
    report_paths.append(report_file)

print(f'Generated {len(report_paths)} gprof report(s):')
for report_file in report_paths:
    print(report_file)

Generated 2 gprof report(s):
/home/bbzze2/ATLAS_sun_model/synthe_1/gprof_spectrv.txt
/home/bbzze2/ATLAS_sun_model/synthe_2/gprof_spectrv.txt


### Reruns

`atlas.synthe` refuses to start if previous `synthe_*` batch directories exist. To preserve prior output, this notebook does not remove them automatically. Archive or explicitly remove the old SYNTHE batch directories before rerunning.